# 18 - Payments: Local Data Is Not the Source of Truth

A transaction's true state lives across the processor, the
network, and the local ledger — and they disagree in real time.
The local record says "settled" while the network says
"pending reversal". Correctness comes from reconciliation, and
WHICH source wins depends on the QUESTION:

| Question (purpose) | Authoritative source |
| --- | --- |
| `fraud.detection` | `network_settlements` |
| `operations.support` (customer balance) | `payment_transactions` |
| `compliance.audit` (fees) | `processor_settlements` |
| `analytics.reporting` | reconciled rows only |

That authority matrix is an ORGANIZATIONAL JUDGMENT — exactly
what Metatate serves. The platform never computes reconciliation;
it tells the agent which source wins and what to verify.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the Customer 360 demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


PRODUCT_DATABASE_TABLES = {"product_usage_events", "support_tickets", "ml_feature_store"}


def asset(table, column=None, schema="public", database=None):
    resolved_database = database or (
        "product" if table in PRODUCT_DATABASE_TABLES else "master"
    )
    ref = {"database": resolved_database, "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## 1. Meaning first: three systems, three truths


In [ ]:
local_meaning = client.inspect_data_meaning(
    ref=asset("payment_transactions", "settlement_state", schema="finance"),
)
network_meaning = client.inspect_data_meaning(
    ref=asset("network_settlements", "settlement_state", schema="finance"),
)
print("local  :", local_meaning["meaning"])
print("network:", network_meaning["meaning"])


## 2. Same transaction, three questions


In [ ]:
authority_cases = [
    ("fraud / network", dict(
        asset=asset("network_settlements", schema="finance"),
        use="investigate suspected fraud on a settled transaction",
        scenario_key="purpose.allowed_use",
        purpose_key="fraud.detection",
    )),
    ("fraud / local ledger", dict(
        asset=asset("payment_transactions", schema="finance"),
        use="investigate suspected fraud on a settled transaction",
        scenario_key="purpose.allowed_use",
        purpose_key="fraud.detection",
    )),
    ("balance / local ledger", dict(
        asset=asset("payment_transactions", schema="finance"),
        use="answer a customer-facing balance question",
        scenario_key="purpose.allowed_use",
        purpose_key="operations.support",
    )),
    ("fees / processor", dict(
        asset=asset("processor_settlements", schema="finance"),
        use="reconcile processor fees for the audit trail",
        scenario_key="purpose.allowed_use",
        purpose_key="compliance.audit",
    )),
]

answers = {}
for label, arguments in authority_cases:
    answer = client.authorize_use(**arguments)
    answers[label] = answer
    print(f"{label:24} -> {answer_label(answer)}")


The SAME fraud question answers differently by SOURCE: the
network table allows, the local ledger fails closed to review —
with the guidance naming where the authoritative state lives.


## 3. Analytics is conditional on reconciliation


In [ ]:
reconciled = client.authorize_use(
    asset("payment_transactions", schema="finance"),
    use="report revenue analytics over payment transactions",
    scenario_key="quality.reliability",
    purpose_key="analytics.reporting",
)
condition = next(
    (c for c in reconciled.get("conditions", [])
     if c.get("kind") == "ai_restriction"),
    {},
)
print(answer_label(reconciled), "->", condition.get("requirement", ""))


## 4. The wrong-source query gets caught


In [ ]:
wrong_source = client.validate_query_context(
    "SELECT transaction_id, settlement_state FROM master.finance.payment_transactions WHERE settlement_state <> 'reconciled'",
    scenario_key="purpose.allowed_use",
    default_database="master", default_schema="public",
    purpose_key="fraud.detection",
)
right_source = client.validate_query_context(
    "SELECT transaction_id, settlement_state FROM master.finance.network_settlements",
    scenario_key="purpose.allowed_use",
    default_database="master", default_schema="public",
    purpose_key="fraud.detection",
)
print("fraud on local ledger ->", wrong_source.get("verdict"), "/", wrong_source.get("state"))
print("fraud on network      ->", right_source.get("verdict"), "/", right_source.get("state"))


Authority is served as judgment, not computed from data: the
policies carry which source wins per question, the conditions
carry what the agent must verify, and every answer cites the
governing policy. Reconciliation itself remains the estate's
job — Metatate makes the authority explicit and auditable.
